In [48]:
import pandas as pd
import numpy as np

# Load the file and convert the '?' characters directly into standard missing values (NaN)
df = pd.read_csv('data/dirty_data.csv', na_values=['?'])

# Replace the '-inf' strings with actual NaN values so they can be cleaned
df.replace('-inf', np.nan, inplace=True)

# Check the structure and see where your gaps are
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               765 non-null    object 
 1   station            398 non-null    object 
 2   PRCP               765 non-null    float64
 3   SNOW               577 non-null    float64
 4   SNWD               577 non-null    float64
 5   TMAX               765 non-null    float64
 6   TMIN               765 non-null    float64
 7   TOBS               398 non-null    float64
 8   WESF               11 non-null     float64
 9   inclement_weather  408 non-null    object 
dtypes: float64(7), object(3)
memory usage: 59.9+ KB


,date,station,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
0,2018-01-01T00:00:00,NaN,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
1,2018-01-01T00:00:00,NaN,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
2,2018-01-01T00:00:00,NaN,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
3,2018-01-02T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-8.3,-16.1,-12.2,NaN,False
4,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False


## Step 1: Examining the Structure

After loading the file with `?` and `-inf` converted to NaN, the structure check reveals several issues:

- **Inconsistent formatting**: The `date` column is stored as a string (`object`) rather than a proper datetime type.
- **Headers**: All 10 column headers are present, but they use mixed case (e.g., `PRCP`, `SNOW`, `inclement_weather`), which is inconsistent.
- **Gaps in the data**:
  - `station` is missing in 367 rows (originally `?`).
  - `SNOW`, `SNWD`, `TOBS`, `WESF`, and `inclement_weather` all contain NaN values.
  - `SNWD` contains `inf` and `-inf` placeholder values instead of real measurements.
  - `TMAX` has 367 rows with the sentinel value `5505` (impossible — likely a "missing" flag).
  - `TMIN` has 367 rows with sentinel `-40` paired with the bogus `TMAX=5505`.
- **Duplicates**: 284 of the 765 rows are exact duplicates that need to be removed.

In [49]:
# Replace inf and -inf in SNWD with NaN (these are placeholders, not real measurements)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# TMAX = 5505 and TMIN = -40 are sentinel values for "missing" — replace with NaN
df.loc[df['TMAX'] == 5505, 'TMAX'] = np.nan
df.loc[df['TMIN'] == -40, 'TMIN'] = np.nan

print("Sentinel values replaced with NaN.")
print(df[['TMAX', 'TMIN', 'SNWD']].describe())

Sentinel values replaced with NaN.
             TMAX        TMIN  SNWD
count  398.000000  398.000000   0.0
mean    15.789196    6.295226   NaN
std     10.838343   10.011548   NaN
min    -11.700000  -17.200000   NaN
25%      6.100000   -1.700000   NaN
50%     14.400000    5.600000   NaN
75%     25.000000   15.600000   NaN
max     35.000000   23.900000   NaN


In [50]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows. Remaining: {len(df)} rows.")

Dropped 284 duplicate rows. Remaining: 481 rows.


In [51]:
# Convert the date column from string to proper datetime
df['date'] = pd.to_datetime(df['date'])

# Standardize all column names to lowercase for consistency
df.columns = df.columns.str.lower()

print("Updated dtypes:")
print(df.dtypes)

Updated dtypes:
date                 datetime64[ns]
station                      object
prcp                        float64
snow                        float64
snwd                        float64
tmax                        float64
tmin                        float64
tobs                        float64
wesf                        float64
inclement_weather            object
dtype: object


In [52]:
# Rows where station is NaN were the '?' rows with bogus temperature sentinels.
# Since they have no valid measurements, drop them.
before = len(df)
df = df.dropna(subset=['station']).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows that had no station identifier.")

# SNWD is now 100% NaN after removing the inf placeholders — drop the column
if df['snwd'].isna().all():
    df = df.drop(columns=['snwd'])
    print("Dropped 'snwd' column (was entirely NaN after cleaning).")

Dropped 232 rows that had no station identifier.
Dropped 'snwd' column (was entirely NaN after cleaning).


## Step 2: Convert the Cleaned Data Into a Tabular Format

The cleaned DataFrame is now in a structured tabular format with consistent column names, proper data types, and meaningful values. Below is a preview, followed by an export to a clean CSV file.

In [53]:
# Display the cleaned tabular data
print(f"Cleaned dataset shape: {df.shape}")
df.head(10)

Cleaned dataset shape: (249, 9)


,date,station,prcp,snow,tmax,tmin,tobs,wesf,inclement_weather
0,2018-01-02,GHCND:USC00280907,0.0,0.0,-8.3,-16.1,-12.2,NaN,False
1,2018-01-03,GHCND:USC00280907,0.0,0.0,-4.4,-13.9,-13.3,NaN,False
2,2018-01-05,GHCND:USC00280907,14.2,127.0,-4.4,-13.9,-13.9,NaN,True
3,2018-01-06,GHCND:USC00280907,0.0,0.0,-10.0,-15.6,-15.0,NaN,False
4,2018-01-07,GHCND:USC00280907,0.0,0.0,-11.7,-17.2,-16.1,NaN,False
5,2018-01-08,GHCND:USC00280907,0.0,0.0,-7.8,-16.7,-8.3,NaN,False
6,2018-01-10,GHCND:USC00280907,0.0,0.0,5.0,-7.8,-7.8,NaN,False
7,2018-01-11,GHCND:USC00280907,0.0,0.0,4.4,-7.8,1.1,NaN,False
8,2018-01-12,GHCND:USC00280907,1.3,0.0,9.4,0.6,7.8,NaN,False
9,2018-01-14,GHCND:USC00280907,0.0,0.0,2.2,-11.1,-11.1,NaN,False


In [54]:
# Save the cleaned data to a new CSV file
df.to_csv('data/cleaned_data.csv', index=False)
print("Cleaned data saved to data/cleaned_data.csv")

Cleaned data saved to data/cleaned_data.csv


## Step 3: Validate the Table

Validation confirms three things: column names are consistent, each column has the correct data type, and missing values have been addressed appropriately.

In [55]:
print("=== Column Names ===")
print(list(df.columns))

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values per Column ===")
print(df.isna().sum())

print("\n=== Duplicate Rows ===")
print(f"Duplicates remaining: {df.duplicated().sum()}")

print("\n=== Summary Statistics ===")
df.describe()

=== Column Names ===
['date', 'station', 'prcp', 'snow', 'tmax', 'tmin', 'tobs', 'wesf', 'inclement_weather']

=== Data Types ===
date                 datetime64[ns]
station                      object
prcp                        float64
snow                        float64
tmax                        float64
tmin                        float64
tobs                        float64
wesf                        float64
inclement_weather            object
dtype: object

=== Missing Values per Column ===
date                   0
station                0
prcp                   0
snow                   5
tmax                   0
tmin                   0
tobs                   0
wesf                 249
inclement_weather      0
dtype: int64

=== Duplicate Rows ===
Duplicates remaining: 0

=== Summary Statistics ===


,date,prcp,snow,tmax,tmin,tobs,wesf
count,249,249.000000,244.000000,249.000000,249.000000,249.000000,0.0
mean,2018-06-30 23:36:52.048192768,5.527309,2.967213,16.004819,6.371486,8.712048,NaN
min,2018-01-02 00:00:00,0.000000,0.000000,-11.700000,-17.200000,-16.100000,NaN
25%,2018-03-27 00:00:00,0.000000,0.000000,6.700000,-1.700000,0.000000,NaN
50%,2018-07-07 00:00:00,0.000000,0.000000,14.400000,5.600000,8.300000,NaN
75%,2018-09-30 00:00:00,5.600000,0.000000,26.100000,15.600000,17.800000,NaN
max,2018-12-31 00:00:00,61.700000,178.000000,35.000000,23.900000,26.100000,NaN
std,NaN,10.665197,20.030608,11.000615,10.157809,9.936468,NaN
